In [1]:
import pandas as pd
import numpy as np

In [2]:
# 1) Gene Expression (RNA-seq)
expr = pd.read_csv("HiSeqV2", sep="\t")

# 2) Clinical data
clinical = pd.read_csv("phenotype.sampleMap_LUNG_clinicalMatrix", sep="\t")

# 3) Protein expression (RPPA)
rppa = pd.read_csv("protein expression (RPPA_RBN)", sep="\t")

# 4) Survival data
survival = pd.read_csv("survival.txt", sep="\t")

# 5) Mutation data
mutation = pd.read_csv("mutation (mc3_gene_level).txt", sep="\t")   # adjust name if needed


In [3]:
expr = expr.set_index("sample")   # gene names as index
expr_T = expr.T                   # transpose

expr_T.reset_index(inplace=True)
expr_T.rename(columns={"index": "patient_id"}, inplace=True)

print("Gene expression shape:", expr_T.shape)


Gene expression shape: (1129, 20531)


In [4]:
clinical.rename(columns={"sample": "patient_id"}, inplace=True)

print("Clinical shape:", clinical.shape)


Clinical shape: (1299, 134)


In [5]:
rppa = rppa.set_index("sample")
rppa_T = rppa.T

rppa_T.reset_index(inplace=True)
rppa_T.rename(columns={"index": "patient_id"}, inplace=True)

print("Protein expression shape:", rppa_T.shape)


Protein expression shape: (432, 132)


In [6]:
survival.rename(columns={"sample": "patient_id"}, inplace=True)

survival = survival[["patient_id", "OS", "OS.time"]]  # OS = dead/alive, OS.time = days

print("Survival shape:", survival.shape)


Survival shape: (1145, 3)


In [7]:
print(mutation.columns)

Index(['xena_sample', 'TCGA-05-4244-01', 'TCGA-05-4249-01', 'TCGA-05-4250-01',
       'TCGA-05-4382-01', 'TCGA-05-4384-01', 'TCGA-05-4389-01',
       'TCGA-05-4390-01', 'TCGA-05-4395-01', 'TCGA-05-4396-01',
       ...
       'TCGA-NK-A5CX-01', 'TCGA-NK-A5D1-01', 'TCGA-NK-A7XE-01',
       'TCGA-O2-A52N-01', 'TCGA-O2-A52Q-01', 'TCGA-O2-A52S-01',
       'TCGA-O2-A52V-01', 'TCGA-O2-A52W-01', 'TCGA-O2-A5IB-01',
       'TCGA-XC-AA0X-01'],
      dtype='object', length=994)


In [8]:
# Load mutation file
mutation = pd.read_csv("mutation (mc3_gene_level).txt", sep="\t")

# Set gene names as index
mutation = mutation.set_index("xena_sample")

# Transpose: rows = patients, columns = genes
mutation_T = mutation.T

# Reset index to create patient_id column
mutation_T.reset_index(inplace=True)
mutation_T.rename(columns={"index": "patient_id"}, inplace=True)

print("Mutation shape:", mutation_T.shape)


Mutation shape: (993, 40544)


In [9]:
cancer_genes = ["TP53", "EGFR", "KRAS", "ALK", "BRAF", "MET", "PIK3CA"]

mutation_T = mutation_T[["patient_id"] + cancer_genes]


In [10]:
expr_T["patient_id"] = expr_T["patient_id"].str[:12]
clinical["patient_id"] = clinical["patient_id"].str[:12]
rppa_T["patient_id"] = rppa_T["patient_id"].str[:12]
survival["patient_id"] = survival["patient_id"].str[:12]
mutation_T["patient_id"] = mutation_T["patient_id"].str[:12]


In [11]:
# Rename mutation columns to avoid conflicts
mutation_cols = mutation_T.columns.tolist()

# Keep patient_id as is, rename others
new_cols = ["patient_id"] + [gene + "_mut" for gene in mutation_cols if gene != "patient_id"]

mutation_T.columns = new_cols

print(mutation_T.columns)


Index(['patient_id', 'TP53_mut', 'EGFR_mut', 'KRAS_mut', 'ALK_mut', 'BRAF_mut',
       'MET_mut', 'PIK3CA_mut'],
      dtype='object')


In [12]:
def normalize_tcga_id(df, col="patient_id"):
    df[col] = df[col].astype(str).str[:12]
    return df

expr_T = normalize_tcga_id(expr_T)
clinical = normalize_tcga_id(clinical)
rppa_T = normalize_tcga_id(rppa_T)
survival = normalize_tcga_id(survival)
mutation_T = normalize_tcga_id(mutation_T)


In [13]:
print("expr:", len(set(expr_T["patient_id"])))
print("clinical:", len(set(clinical["patient_id"])))
print("rppa:", len(set(rppa_T["patient_id"])))
print("survival:", len(set(survival["patient_id"])))
print("mutation:", len(set(mutation_T["patient_id"])))

common_ids = set(expr_T["patient_id"]) & set(clinical["patient_id"]) & set(rppa_T["patient_id"]) & set(survival["patient_id"]) & set(mutation_T["patient_id"])
print("Common patients:", len(common_ids))


expr: 1018
clinical: 1025
rppa: 432
survival: 1023
mutation: 993
Common patients: 0


In [14]:
data = pd.merge(expr_T, clinical, on="patient_id", how="inner")
print("After expr + clinical:", data.shape)

data = pd.merge(data, survival, on="patient_id", how="inner")
print("After + survival:", data.shape)

data = pd.merge(data, rppa_T, on="patient_id", how="inner")
print("After + rppa:", data.shape)

data = pd.merge(data, mutation_T, on="patient_id", how="inner")
print("After + mutation:", data.shape)


After expr + clinical: (0, 20664)
After + survival: (0, 20666)
After + rppa: (0, 20797)
After + mutation: (0, 20804)


In [15]:
print(clinical.columns.tolist())


['sampleID', 'ABSOLUTE_Ploidy', 'ABSOLUTE_Purity', 'AKT1', 'ALK_translocation', 'BRAF', 'CBL', 'CTNNB1', 'Canonical_mut_in_KRAS_EGFR_ALK', 'Cnncl_mt_n_KRAS_EGFR_ALK_RET_ROS1_BRAF_ERBB2_HRAS_NRAS_AKT1_MAP2', 'EGFR', 'ERBB2', 'ERBB4', 'Estimated_allele_fraction_of_a_clonal_varnt_prsnt_t_1_cpy_pr_cll', 'Expression_Subtype', 'HRAS', 'KRAS', 'MAP2K1', 'MET', 'NRAS', 'PIK3CA', 'PTPN11', 'Pathology', 'Pathology_Updated', 'RET_translocation', 'ROS1_translocation', 'STK11', 'WGS_as_of_20120731_0_no_1_yes', '_INTEGRATION', '_PANCAN_CNA_PANCAN_K8', '_PANCAN_Cluster_Cluster_PANCAN', '_PANCAN_DNAMethyl_PANCAN', '_PANCAN_RPPA_PANCAN_K8', '_PANCAN_UNC_RNAseq_PANCAN_K16', '_PANCAN_miRNA_PANCAN', '_PANCAN_mutation_PANCAN', '_PATIENT', '_cohort', '_primary_disease', '_primary_site', 'additional_pharmaceutical_therapy', 'additional_radiation_therapy', 'additional_surgery_locoregional_procedure', 'additional_surgery_metastatic_procedure', 'age_at_initial_pathologic_diagnosis', 'anatomic_neoplasm_subdivisi

In [16]:
print(clinical.head())


          sampleID  ABSOLUTE_Ploidy  ABSOLUTE_Purity  AKT1 ALK_translocation  \
0  TCGA-05-4244-01              NaN              NaN   NaN               NaN   
1  TCGA-05-4249-01             3.77             0.46  none               NaN   
2  TCGA-05-4250-01              NaN              NaN   NaN               NaN   
3  TCGA-05-4382-01              NaN              NaN  none               NaN   
4  TCGA-05-4384-01             2.04             0.48  none               NaN   

      BRAF   CBL   CTNNB1 Canonical_mut_in_KRAS_EGFR_ALK  \
0      NaN   NaN      NaN                            NaN   
1  p.A762E  none     none                              Y   
2      NaN   NaN      NaN                            NaN   
3  p.L613F  none     none                              N   
4     none  none  p.F777S                              N   

  Cnncl_mt_n_KRAS_EGFR_ALK_RET_ROS1_BRAF_ERBB2_HRAS_NRAS_AKT1_MAP2  ...  \
0                                                NaN                ...   
1       

In [17]:
for col in clinical.columns:
    sample_vals = clinical[col].astype(str).head(10).tolist()
    if any("TCGA" in v for v in sample_vals):
        print("POSSIBLE ID COLUMN:", col)
        print(sample_vals[:5])


POSSIBLE ID COLUMN: sampleID
['TCGA-05-4244-01', 'TCGA-05-4249-01', 'TCGA-05-4250-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01']
POSSIBLE ID COLUMN: _INTEGRATION
['TCGA-05-4244-01', 'TCGA-05-4249-01', 'TCGA-05-4250-01', 'TCGA-05-4382-01', 'TCGA-05-4384-01']
POSSIBLE ID COLUMN: _PATIENT
['TCGA-05-4244', 'TCGA-05-4249', 'TCGA-05-4250', 'TCGA-05-4382', 'TCGA-05-4384']
POSSIBLE ID COLUMN: _cohort
['TCGA Lung Cancer (LUNG)', 'TCGA Lung Cancer (LUNG)', 'TCGA Lung Cancer (LUNG)', 'TCGA Lung Cancer (LUNG)', 'TCGA Lung Cancer (LUNG)']
POSSIBLE ID COLUMN: bcr_followup_barcode
['nan', 'TCGA-05-4249-F36327', 'nan', 'TCGA-05-4382-F36329', 'TCGA-05-4384-F36330']
POSSIBLE ID COLUMN: bcr_patient_barcode
['TCGA-05-4244', 'TCGA-05-4249', 'TCGA-05-4250', 'TCGA-05-4382', 'TCGA-05-4384']
POSSIBLE ID COLUMN: bcr_sample_barcode
['TCGA-05-4244-01A', 'TCGA-05-4249-01A', 'TCGA-05-4250-01A', 'TCGA-05-4382-01A', 'TCGA-05-4384-01A']
POSSIBLE ID COLUMN: pathology_report_file_name
['TCGA-05-4244.3a844132-f813-4d8e-8f7d-

In [18]:
clinical["patient_id"] = clinical["sampleID"].astype(str).str[:12]


In [19]:
print("expr sample IDs:", expr_T["patient_id"].head(5).tolist())
print("clinical sample IDs:", clinical["patient_id"].head(5).tolist())

common_ids = set(expr_T["patient_id"]) & set(clinical["patient_id"])
print("Common expr + clinical patients:", len(common_ids))


expr sample IDs: ['TCGA-NJ-A4YP', 'TCGA-18-3417', 'TCGA-22-4613', 'TCGA-90-7769', 'TCGA-62-8397']
clinical sample IDs: ['TCGA-05-4244', 'TCGA-05-4249', 'TCGA-05-4250', 'TCGA-05-4382', 'TCGA-05-4384']
Common expr + clinical patients: 1018


In [20]:
data = pd.merge(expr_T, clinical, on="patient_id", how="inner")
print("After expr + clinical:", data.shape)


After expr + clinical: (1449, 20664)


In [21]:
def clean_tcga_id(df, col="patient_id"):
    df[col] = df[col].astype(str)
    df[col] = df[col].str.strip()
    df[col] = df[col].str.upper()
    df[col] = df[col].str.replace(".", "-", regex=False)
    df[col] = df[col].str[:12]   # TCGA-XX-XXXX
    return df

expr_T = clean_tcga_id(expr_T)
clinical = clean_tcga_id(clinical)
survival = clean_tcga_id(survival)
rppa_T = clean_tcga_id(rppa_T)
mutation_T = clean_tcga_id(mutation_T)


In [22]:
# Remove duplicate gene columns between datasets
expr_cols = set(expr_T.columns)
rppa_T = rppa_T[[c for c in rppa_T.columns if c not in expr_cols or c == "patient_id"]]
mutation_T = mutation_T[[c for c in mutation_T.columns if c not in expr_cols or c == "patient_id"]]


In [23]:
data = expr_T.copy()   # base dataset

data = pd.merge(data, clinical, on="patient_id", how="left")
print("After clinical merge:", data.shape)

data = pd.merge(data, survival, on="patient_id", how="left")
print("After survival merge:", data.shape)

data = pd.merge(data, rppa_T, on="patient_id", how="left")
print("After RPPA merge:", data.shape)

data = pd.merge(data, mutation_T, on="patient_id", how="left")
print("After mutation merge:", data.shape)


After clinical merge: (1449, 20664)
After survival merge: (1787, 20666)
After RPPA merge: (1787, 20765)
After mutation merge: (1787, 20772)


In [24]:
data.fillna(0, inplace=True)


In [25]:
data.to_csv("lung_cancer_final_dataset.csv", index=False)
print("✅ FINAL DATASET CREATED!")
print("Final shape:", data.shape)


✅ FINAL DATASET CREATED!
Final shape: (1787, 20772)


In [26]:
lung_cancer_genes = [
    "TP53", "EGFR", "KRAS", "ALK", "BRAF", "MET", "PIK3CA", "ROS1", "RET",
    "ERBB2", "PTEN", "RB1", "MYC", "CDKN2A", "FGFR1", "FGFR2", "FGFR3",
    "SMAD4", "STK11", "KEAP1", "NFE2L2"
]


In [27]:
selected_gene_cols = [g for g in lung_cancer_genes if g in data.columns]


In [28]:
clinical_cols = [
    "patient_id",
    "age_at_diagnosis",
    "gender",
    "tumor_stage",
    "smoking_status",
    "histological_type"
]

clinical_cols = [c for c in clinical_cols if c in data.columns]


In [29]:
mutation_cols = [c for c in data.columns if c.endswith("_mut")]


In [30]:
rppa_cols = [c for c in data.columns if c.startswith("RPPA")]


In [31]:
label_cols = ["OS", "OS.time"]
label_cols = [c for c in label_cols if c in data.columns]


In [32]:
final_cols = ["patient_id"] + selected_gene_cols + mutation_cols + clinical_cols + label_cols
final_cols = list(dict.fromkeys(final_cols))  # remove duplicates

genai_data = data[final_cols]

print("GenAI dataset shape:", genai_data.shape)


GenAI dataset shape: (1787, 26)


In [33]:
genai_data = pd.get_dummies(genai_data, drop_first=True)


In [34]:
import sklearn
from sklearn.preprocessing import StandardScaler

feature_cols = [c for c in genai_data.columns if c not in ["patient_id", "OS", "OS.time"]]

scaler = StandardScaler()
genai_data[feature_cols] = scaler.fit_transform(genai_data[feature_cols])


In [35]:
genai_data["risk_label"] = (genai_data["OS.time"] < 365*3).astype(int)


C:\Users\Pruthvi\AppData\Local\Temp\ipykernel_27456\3877007222.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  genai_data["risk_label"] = (genai_data["OS.time"] < 365*3).astype(int)


In [36]:
genai_data.to_csv("lung_cancer_genai_dataset.csv", index=False)
print("✅ FINAL GENAI DATASET CREATED!")


✅ FINAL GENAI DATASET CREATED!


In [37]:
gene_list = selected_gene_cols

protein_dataset = pd.DataFrame({"gene": gene_list})
protein_dataset.to_csv("lung_cancer_genai_proteins.csv", index=False)

print("✅ Protein dataset created!")


✅ Protein dataset created!


In [38]:
!pip install monai

In [39]:
import glob
images = sorted(glob.glob(r"D:/major/output_nifti/*.nii.gz"))
# If you have masks: labels = sorted(glob.glob(r"D:\major\output_masks\*.nii.gz"))
data_dicts = [{"image": img} for img in images]  # for inference only

In [40]:
import monai
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Spacingd, ScaleIntensityRanged,
    CropForegroundd, SpatialPadd, RandCropByPosNegLabeld, ToTensord
)
from monai.data import Dataset, DataLoader
from monai.utils import set_determinism
import os

# Set seed for reproducibility
set_determinism(seed=42)

# Paths (update to your dataset)
data_dir = "D:/major/output_nifti"
images = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if 'image' in f])
labels = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if 'mask' in f])
data_dicts = [{"image": img, "label": lbl} for img, lbl in zip(images, labels)]

# Split: 80% train, 20% val
train_data = data_dicts[:int(0.8 * len(data_dicts))]
val_data = data_dicts[int(0.8 * len(data_dicts)):]

# Transforms for training
train_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),  # Resample to 1mm isotropic
    ScaleIntensityRanged(keys=["image"], a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),  # Normalize CT Hounsfield units
    CropForegroundd(keys=["image", "label"], source_key="image"),  # Remove background
    SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 128)),  # Pad to fixed size
    RandCropByPosNegLabeld(keys=["image", "label"], label_key="label", spatial_size=(96, 96, 96), pos=1.0, neg=1.0, num_samples=4),
    ToTensord(keys=["image", "label"])
])

# Validation transforms (no augmentation)
val_transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Spacingd(keys=["image", "label"], pixdim=(1.0, 1.0, 1.0), mode=("bilinear", "nearest")),
    ScaleIntensityRanged(keys=["image"], a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    CropForegroundd(keys=["image", "label"], source_key="image"),
    SpatialPadd(keys=["image", "label"], spatial_size=(128, 128, 128)),
    ToTensord(keys=["image", "label"])
])

# Datasets and Loaders
train_ds = Dataset(data=train_data, transform=train_transforms)
val_ds = Dataset(data=val_data, transform=val_transforms)
train_loader = DataLoader(train_ds, batch_size=2, shuffle=True, num_workers=4)
val_loader = DataLoader(val_ds, batch_size=1, num_workers=4)

ValueError: num_samples should be a positive integer value, but got num_samples=0